# Build MDE webapp data (`docs/data/*.json`)

Assembles JSON files the static viewer (`docs/`) consumes:

- `docs/data/mde.json` — points (x, y, cluster ids, n_DEGs, edist) + manual cluster labels
- `docs/data/degs.json` — top-10 up/down DEGs per perturbation (for the click side panel)

**Inputs**
- `KOLF_Perturbation_Atlas_Analysis/output_files/KOLF_Pan_Genome_Filtered_Replogle_Pipeline_Modified_MDE_clusters_annotated.xlsx` — sheet `MDE` has 1,656 perts (we drop the single NTC reference point → 1,655 for the viewer).
- `KOLF_Perturbation_Atlas_Analysis/output_files/KOLF_Pan_Genome_Energy_Test_Gene_Level.csv` — `gene_target, n_DEGs, edist, ...` per pert.
- `KOLF_Perturbation_Atlas_Analysis/output_files/KOLF_Pan_Genome_DEGs_per_Strong_Perturbation.csv` — wide-format DEG table with `{GENE}_DEGs`, `{GENE}_L2FC`, `{GENE}_Adj` triplets.
- `MANUAL_LEIDEN` / `MANUAL_HDBSCAN` (below) — 62 and 30 manually curated cluster names.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path('/tscc/projects/ps-malilab/ydoctor/KOLF_Perturbation_Atlas')
MDE_XLSX  = ROOT / 'KOLF_Perturbation_Atlas_Analysis/output_files/KOLF_Pan_Genome_Filtered_Replogle_Pipeline_Modified_MDE_clusters_annotated.xlsx'
EDIST_CSV = ROOT / 'KOLF_Perturbation_Atlas_Analysis/output_files/KOLF_Pan_Genome_Energy_Test_Gene_Level.csv'
DEGS_CSV  = ROOT / 'KOLF_Perturbation_Atlas_Analysis/output_files/KOLF_Pan_Genome_DEGs_per_Strong_Perturbation.csv'
OUT_DIR   = ROOT / 'docs/data'
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
MANUAL_LEIDEN = [
    "", "", "", "", "",
    "npBAF complex; Cohesin complex; PcG complex", "",
    "Transcription Regulation", "Mitochondrial Translation", "",
    "Mitochondrial Translation", "", "DNA Repair", "",
    "Mitochondrial Translation", "Neddylation", "",
    "P-Body; NuA4 Complex",
    "SAGA complex; Prefoldin-like complex; Mediator complex", "",
    "Oxidative Phosphorylation", "",
    "Mitochondrial RNA metabolic process", "",
    "U2 snRNP", "Ragulator Complex; Respirasome", "",
    "Lysosome", "", "tRNA Aminoacylation", "",
    "Mitochondrial Membrane", "TFIID complex", "Pluripotency Signaling", "",
    "Cortical Cytoskeleton",
    "m6A Transferase; INO80 complex; CREBBP/EP300",
    "Transcription Regulation",
    "Mitochondrial Membrane; Apoptosome",
    "snRNA Transcription; Cajal Body; SMN complex",
    "SIN3 Complex; Histone Methyltransferase",
    "Translation Initiation", "",
    "Endoplasmic Reticulum", "Membrane Trafficking", "Mitochondrial Translation",
    "Ubiquitin Complex; Methyltransferase complex",
    "AP-2 adaptor complex", "tRNA processing", "", "",
    "POU domain", "", "",
    "DNA methylation", "Methylosome", "",
    "Centromeric Region", "Protein Ufmylation", "TGF-beta Signaling", "",
    "Mitochondrial Translation",
]
MANUAL_HDBSCAN = [
    "AP2 complex", "RNA Pol II complex", "Cortical Cytoskeleton",
    "Cohesin complex", "npBAF complex; PcG complex", "INO80 complex",
    "Hippo-YAP Signaling", "Heparan Sulfate Proteoglycan", "",
    "Integrator complex", "Neddylation", "RNA degradation", "DNA Methylation",
    "SIN3 complex", "", "DNA Methylation", "",
    "snRNA Transcription; Cajal Body; SMN complex", "TGF-beta Signaling",
    "CCR4-NOT complex", "POU domain", "Transcription Regulation", "",
    "SAGA complex", "Mediator complex", "TFIID complex",
    "Mitochondrial Translation", "Mitochondrial Transcription",
    "Mitochondrial Membrane: Apoptosome", "Ragulator complex; Late Endosomes",
]
assert len(MANUAL_LEIDEN) == 62 and len(MANUAL_HDBSCAN) == 30

In [ ]:
# --- Points: MDE coords + cluster ids + edist + n_DEGs ---
mde = pd.read_excel(MDE_XLSX, sheet_name='MDE').rename(columns={
    'gene_target': 'gene', 'leiden cluster': 'leiden', 'hdbscan cluster': 'hdbscan',
})

etest = pd.read_csv(EDIST_CSV)[['gene_target', 'n_DEGs', 'edist']]
mde = (
    mde.merge(etest, how='left', left_on='gene', right_on='gene_target')
       .drop(columns=['gene_target'])
)
mde['n_DEGs'] = mde['n_DEGs'].fillna(-1).astype(int)

# Drop NTC reference point (has no perturbation stats)
mde = mde[mde['gene'] != 'NTC'].reset_index(drop=True)
print(f'{len(mde)} perts after dropping NTC')

In [ ]:
points = [
    {
        'g': r.gene,
        'x': round(float(r.x), 3),
        'y': round(float(r.y), 3),
        'l': int(r.leiden),
        'h': int(r.hdbscan),
        'n': int(r.n_DEGs),
        'e': None if pd.isna(r.edist) else round(float(r.edist), 3),
    }
    for r in mde.itertuples(index=False)
]

payload = {
    'points': points,
    'leiden_labels':  {str(i): MANUAL_LEIDEN[i]  for i in range(62)},
    'hdbscan_labels': {str(i): MANUAL_HDBSCAN[i] for i in range(30)},
}
with open(OUT_DIR / 'mde.json', 'w') as f:
    json.dump(payload, f, separators=(',', ':'))
print(f"wrote mde.json ({(OUT_DIR / 'mde.json').stat().st_size / 1024:.1f} KB, {len(points)} points)")

In [ ]:
# --- Top-N up/down DEGs per pert ---
# Wide-format CSV: for each pert GENE, columns {GENE}_DEGs, {GENE}_L2FC, {GENE}_Adj.
degs = pd.read_csv(DEGS_CSV, low_memory=False)
perts_in_degs = set(c.rsplit('_', 1)[0] for c in degs.columns if c.endswith('_DEGs'))

K = 10
ADJ_THRESH = 0.05

def top_degs(pert: str):
    sub = degs[[f'{pert}_DEGs', f'{pert}_L2FC', f'{pert}_Adj']].dropna()
    sub.columns = ['g', 'lfc', 'adj']
    sub = sub[sub['adj'] <= ADJ_THRESH]
    up = sub.sort_values('lfc', ascending=False).head(K)
    dn = sub.sort_values('lfc', ascending=True).head(K)
    return (
        [{'g': r.g, 'lfc': round(float(r.lfc), 2)} for r in up.itertuples(index=False)],
        [{'g': r.g, 'lfc': round(float(r.lfc), 2)} for r in dn.itertuples(index=False)],
    )

mde_genes = set(mde['gene'])
degs_payload = {}
for g in mde_genes:
    if g in perts_in_degs:
        up, dn = top_degs(g)
        degs_payload[g] = {'up': up, 'dn': dn}

with open(OUT_DIR / 'degs.json', 'w') as f:
    json.dump(degs_payload, f, separators=(',', ':'))
print(f"wrote degs.json ({(OUT_DIR / 'degs.json').stat().st_size / 1024:.1f} KB, {len(degs_payload)}/{len(mde_genes)} perts covered)")

## Preview locally

```bash
cd docs && python -m http.server 8000
# then open http://localhost:8000
```

## Deploy on GitHub Pages

1. Commit `docs/` and push to `main`.
2. GitHub → repo **Settings → Pages**: set **Source = Deploy from a branch**, **Branch = `main`**, **folder `/docs`**. Save.
3. Site is live at `https://y-doctor.github.io/KOLF2.1J_Perturbation_Cell_Atlas/`.